# Tutorial: Hola ADK

**KEEPCODING · AI Engineering · Clase 5 · Módulo M2: Agent Development Kit (ADK)**

En este laboratorio damos nuestros primeros pasos con el **Agent Development Kit (ADK)** de Google, un toolkit de código abierto para construir agentes de IA.

## Para quién es

Alumnado del curso que ya conoce Python básico y la idea de "tool calling" (un LLM que llama a funciones). No hace falta experiencia previa con ADK.

## Qué vas a aprender

- Los cuatro primitivos de ADK: **Agent**, **Tool**, **Runner** y **Session**.
- Cómo una función Python tipada se convierte en una herramienta automáticamente.
- Que **ADK es agnóstico de modelo**: el mismo agente funciona con Gemini o con OpenAI cambiando solo un argumento.
- Cómo leer el flujo de eventos (`Event`) que produce un agente al ejecutarse.

## Qué necesitas

- **Google Colab** (recomendado): este notebook está pensado para ejecutarse ahí de arriba a abajo.
- Una **`OPENAI_API_KEY`** (obligatoria): el lab corre de principio a fin con OpenAI (`gpt-5-mini` vía LiteLLM).
- **Opcional**: una **`GOOGLE_API_KEY`** si quieres probar también Gemini como alternativa. Si no la tienes, no se te pide y la celda de Gemini se salta sola con un aviso.
- Todo es **local**: no usamos `adk deploy`, ni Vertex AI, ni nada de cloud. Solo el SDK corriendo en el runtime del notebook.

> Nota: este notebook está pensado para ejecutarse de principio a fin. Ejecuta las celdas en orden.

### Por qué importa

ADK es la apuesta de Google para agentes de producción (Gemini + Vertex AI). Conocerlo te da criterio para **elegir framework**: LangChain (M1), ADK (aquí) o tu propio harness (M5) resuelven el mismo problema con trade-offs distintos.

### Conecta con

- **Slides M2** · qué es ADK, primitivos, Vertex AI.
- **M1** · el mismo agente, otro framework (LangChain/LangGraph).
- **M3** · A2A, del mismo ecosistema Google.
- **Vertex AI Agent Engine** · el destino natural para desplegar un agente ADK en producción (fuera del alcance de este lab, que es 100 % local).

### Stack

`google-adk>=2.0` · `litellm` · `gemini-2.5-flash` (Gemini) / `gpt-5-mini` (OpenAI vía LiteLLM).

### Hardware

Cualquier CPU; Colab Free sobra (no usa GPU, todo es API). Necesita `OPENAI_API_KEY`; `GOOGLE_API_KEY` es opcional (solo para la alternativa con Gemini).


## Guion del laboratorio

1. **Setup**: instalar `google-adk` y `litellm`, y configurar las claves de API.
2. **La Tool**: una función Python tipada, `buscar_personaje_onepiece`, que ADK convierte en herramienta.
3. **El Agent**: construir un agente (primero con Gemini, como ejemplo de modelo nativo de Google) que usa esa tool.
4. **Agnóstico de modelo**: cambiar SOLO el modelo a OpenAI con `LiteLlm`. Mismo agente, misma tool.
5. **Runner y Session**: preparar el motor de ejecución y una sesión de conversación.
6. **Ejecutar y leer eventos**: lanzar el agente e iterar el stream de `Event`s.
7. **Recap y retos**.


## Mapa mental: cómo encajan las piezas

Antes de tocar código, este es el dibujo de lo que vamos a montar. Los **cuatro primitivos** de ADK no funcionan sueltos: forman una cadena que va desde tu pregunta hasta una función Python que se ejecuta de verdad.

<svg viewBox="0 0 760 360" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Cómo encajan los 4 primitivos de ADK" style="max-width:760px;width:100%;height:auto;background:#fafafa;border-radius:10px;">
  <defs>
    <marker id="a1" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto">
      <path d="M0,0 L10,5 L0,10 z" fill="#374151"/>
    </marker>
  </defs>
  <style>
    .ttl { font: 600 14px -apple-system, system-ui, sans-serif; fill:#111827; }
    .lbl { font: 400 12px -apple-system, system-ui, sans-serif; fill:#374151; }
    .sml { font: 400 11px -apple-system, system-ui, sans-serif; fill:#6b7280; }
    .hdr { font: 700 16px -apple-system, system-ui, sans-serif; fill:#111827; }
    .box { stroke:#374151; stroke-width:1.5; }
    .arr { stroke:#374151; stroke-width:1.5; fill:none; marker-end:url(#a1); }
  </style>

  <text x="380" y="32" text-anchor="middle" class="hdr">Los 4 primitivos de ADK encajando entre sí</text>

  <!-- User -->
  <circle cx="60" cy="180" r="30" fill="#fef3c7" class="box"/>
  <text x="60" y="186" text-anchor="middle" class="ttl">tú</text>
  <text x="60" y="226" text-anchor="middle" class="sml">usuario</text>

  <!-- Arrow user -> runner -->
  <path d="M92,180 L156,180" class="arr"/>
  <text x="124" y="170" text-anchor="middle" class="sml">pregunta</text>

  <!-- Runner -->
  <rect x="160" y="140" width="140" height="80" rx="10" fill="#dbeafe" class="box"/>
  <text x="230" y="168" text-anchor="middle" class="ttl">Runner</text>
  <text x="230" y="188" text-anchor="middle" class="lbl">motor de</text>
  <text x="230" y="204" text-anchor="middle" class="lbl">ejecución</text>

  <!-- Session above Runner -->
  <rect x="160" y="60" width="140" height="60" rx="10" fill="#fce7f3" class="box"/>
  <text x="230" y="84" text-anchor="middle" class="ttl">Session</text>
  <text x="230" y="104" text-anchor="middle" class="lbl">historial de chat</text>
  <path d="M230,140 L230,120" class="arr"/>
  <text x="250" y="134" class="sml">lee / escribe</text>

  <!-- Arrow runner -> agent -->
  <path d="M300,180 L364,180" class="arr"/>
  <text x="332" y="170" text-anchor="middle" class="sml">ejecuta</text>

  <!-- Agent -->
  <rect x="370" y="120" width="220" height="140" rx="10" fill="#dcfce7" class="box"/>
  <text x="480" y="148" text-anchor="middle" class="ttl">Agent (el cerebro)</text>
  <line x1="384" y1="160" x2="576" y2="160" stroke="#9ca3af" stroke-dasharray="3,3"/>
  <text x="384" y="180" class="lbl">model:</text>
  <text x="445" y="180" class="lbl">"gemini-..." o LiteLlm(...)</text>
  <text x="384" y="202" class="lbl">instruction:</text>
  <text x="465" y="202" class="lbl">prompt de sistema</text>
  <text x="384" y="224" class="lbl">tools:</text>
  <text x="425" y="224" class="lbl">[funcion1, funcion2, ...]</text>
  <text x="384" y="246" class="sml">razona y decide cuando llamar a una tool</text>

  <!-- Arrow agent -> tool -->
  <path d="M590,210 L660,260" class="arr"/>
  <text x="640" y="225" text-anchor="middle" class="sml">llama si</text>
  <text x="640" y="238" text-anchor="middle" class="sml">hace falta</text>

  <!-- Tool -->
  <rect x="640" y="270" width="100" height="70" rx="10" fill="#fee2e2" class="box"/>
  <text x="690" y="296" text-anchor="middle" class="ttl">Tool</text>
  <text x="690" y="316" text-anchor="middle" class="lbl">tu función</text>
  <text x="690" y="330" text-anchor="middle" class="lbl">Python (.py)</text>

  <!-- Return path agent -> user -->
  <path d="M370,260 Q 200,310 90,200" class="arr"/>
  <text x="220" y="320" text-anchor="middle" class="sml">respuesta + stream de Events</text>
</svg>

Léelo de izquierda a derecha:

- **Tú** lanzas una pregunta.
- El **Runner** la recibe, consulta el historial en la **Session** y se la pasa al agente.
- El **Agent** combina un LLM + una instrucción + una lista de **tools**. Decide si necesita una tool y, en ese caso, la **Tool** se ejecuta como cualquier función Python.
- Todo lo que pasa por dentro sale en forma de **stream de Events** que tú puedes leer.

A lo largo del notebook vamos a construir cada caja, de izquierda a derecha.


## 1. Setup

### 1.1 Instalar dependencias

Necesitamos dos paquetes:

- **`google-adk`**: el toolkit de agentes. Trae `Agent`, `Runner`, sesiones, etc.
- **`litellm`**: un adaptador universal de LLMs. ADK lo usa por debajo para hablar con modelos que no son de Google (OpenAI, Anthropic, etc.).

En Colab esta instalación tarda un minuto. Si ves un aviso para reiniciar el runtime, normalmente no hace falta para este lab.


In [1]:
# Instalación silenciosa de las dependencias del lab
# google-genai se instala explícito: ADK lo usa para `from google.genai import types`.
# mcp: las versiones recientes de litellm lo importan en cuanto el agente tiene tools
# (aunque no uses MCP). Sin él, la llamada a OpenAI falla con "No module named 'mcp'".
%pip install -q "google-adk>=2.0" google-genai litellm mcp nest_asyncio  # ADK 2.0 GA mayo 2026

print("Dependencias instaladas.")

Note: you may need to restart the kernel to use updated packages.
Dependencias instaladas.


### 1.2 Configurar las claves de API

ADK lee las claves desde **variables de entorno**. En Colab lo más cómodo es guardarlas en el panel de **Secrets** (icono de la llave a la izquierda) con los nombres `OPENAI_API_KEY` y, opcionalmente, `GOOGLE_API_KEY`.

La celda siguiente intenta leerlas desde Secrets de Colab. Si no estás en Colab y la `OPENAI_API_KEY` no está en el entorno, te la pide por teclado. La `OPENAI_API_KEY` es obligatoria; la `GOOGLE_API_KEY` es opcional y nunca se pide por teclado: si no está, la parte de Gemini se omite.


In [2]:
import os
import getpass


def cargar_clave(nombre: str, obligatoria: bool) -> None:
    """Carga una clave de API en las variables de entorno.

    Intenta leerla desde los Secrets de Colab; si no, la pide por teclado.
    """
    if os.environ.get(nombre):
        print(f"{nombre}: ya configurada.")
        return

    valor = None
    try:
        from google.colab import userdata  # type: ignore

        valor = userdata.get(nombre)
    except Exception:
        valor = None

    if not valor and obligatoria:
        valor = getpass.getpass(f"Introduce tu {nombre}: ")

    if valor:
        os.environ[nombre] = valor
        print(f"{nombre}: configurada.")
    else:
        print(f"{nombre}: no configurada. Es opcional: la parte de Gemini se omitirá y el lab sigue con OpenAI.")


cargar_clave("OPENAI_API_KEY", obligatoria=True)
cargar_clave("GOOGLE_API_KEY", obligatoria=False)


OPENAI_API_KEY: ya configurada.
GOOGLE_API_KEY: no configurada. Es opcional: la parte de Gemini se omitirá y el lab sigue con OpenAI.


### 1.3 Imports

Importamos los primitivos de ADK que vamos a usar. También aplicamos `nest_asyncio`: ADK ejecuta los agentes con `asyncio`, y los notebooks ya tienen un bucle de eventos en marcha. `nest_asyncio` permite anidar ese bucle sin errores.


In [3]:
import warnings

import nest_asyncio

# ADK avisa de features [EXPERIMENTAL] que activa por dentro: ruido para este lab.
warnings.filterwarnings("ignore", message=r"\[EXPERIMENTAL\]")

# Permite usar asyncio dentro del bucle de eventos del notebook
nest_asyncio.apply()

# Primitivos de ADK
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

# Tipos de mensaje compartidos con la SDK de Gemini
from google.genai import types

print("ADK listo para usar.")


ADK listo para usar.


## 2. La Tool: una función Python tipada

Una **Tool** (herramienta) es algo que el agente puede llamar para hacer cosas que un LLM no puede hacer solo: consultar una API, leer una base de datos, calcular.

Lo elegante de ADK: una **función Python normal** ya es una tool. ADK lee:

- el **nombre** de la función,
- las **anotaciones de tipo** de los argumentos,
- el **docstring**,

y genera automáticamente el *schema* que el LLM necesita para saber cuándo y cómo llamarla. No escribimos JSON a mano.

### El hilo friki: One Piece

Nuestra tool será `buscar_personaje_onepiece(nombre)`. Intenta consultar la **API en vivo** de [api-onepiece.com](https://api-onepiece.com) y, si falla (sin red, API caída, etc.), cae a un **diccionario hardcodeado** con varios personajes principales. Así el lab funciona siempre, incluso sin internet.


### 2.1 Datos de respaldo (fallback)

Primero definimos un diccionario con varios personajes. Es la red de seguridad: si la API en vivo no responde, la tool usa estos datos.


In [4]:
# Diccionario de respaldo: personajes principales de One Piece.
# Las claves están en minúsculas para buscar sin importar mayúsculas.
PERSONAJES_FALLBACK = {
    "luffy": {
        "nombre": "Monkey D. Luffy",
        "tripulación": "Piratas de Sombrero de Paja",
        "fruta": "Gomu Gomu no Mi (Hito Hito no Mi, modelo Nika)",
        "recompensa": "3.000.000.000 berries",
        "sueño": "Convertirse en el Rey de los Piratas",
    },
    "zoro": {
        "nombre": "Roronoa Zoro",
        "tripulación": "Piratas de Sombrero de Paja",
        "fruta": "Ninguna",
        "recompensa": "1.111.000.000 berries",
        "sueño": "Ser el mejor espadachín del mundo",
    },
    "nami": {
        "nombre": "Nami",
        "tripulación": "Piratas de Sombrero de Paja",
        "fruta": "Ninguna",
        "recompensa": "366.000.000 berries",
        "sueño": "Dibujar un mapa de todo el mundo",
    },
    "sanji": {
        "nombre": "Vinsmoke Sanji",
        "tripulación": "Piratas de Sombrero de Paja",
        "fruta": "Ninguna",
        "recompensa": "1.032.000.000 berries",
        "sueño": "Encontrar el All Blue",
    },
    "ace": {
        "nombre": "Portgas D. Ace",
        "tripulación": "Piratas de Barbablanca",
        "fruta": "Mera Mera no Mi",
        "recompensa": "550.000.000 berries",
        "sueño": "Que su tripulación lo recuerde con orgullo",
    },
    "shanks": {
        "nombre": "Shanks",
        "tripulación": "Piratas del Pelirrojo",
        "fruta": "Ninguna",
        "recompensa": "4.048.900.000 berries",
        "sueño": "Desconocido",
    },
}

print(f"Personajes de respaldo cargados: {len(PERSONAJES_FALLBACK)}")


Personajes de respaldo cargados: 6


### 2.2 La función tool

Ahora la tool. Fíjate en tres detalles que ADK usa para generar el schema:

- **Tipo del argumento**: `nombre: str`.
- **Docstring claro**: explica qué hace y qué devuelve. El LLM lee esto para decidir cuándo llamarla.
- **Devuelve un `dict`**: por convención en ADK, incluimos una clave `status` (`"ok"` o `"error"`) para que el agente sepa si la llamada fue bien.

In [5]:
import urllib.request
import json
import unicodedata

# Cache en memoria: la lista completa de personajes se descarga una sola vez.
_OP_CACHE = []


def _normalizar(texto: str) -> str:
    """Minúsculas, sin acentos ni puntuación, espacios colapsados."""
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    t = "".join(c if c.isalnum() else " " for c in t.lower())
    return " ".join(t.split())


def _coincide(consulta: str, nombre: str) -> bool:
    """True si 'consulta' identifica a 'nombre'. Tolera puntos, acentos y orden.

    El LLM puede pasar "Luffy" o "Monkey D. Luffy"; la API guarda
    "Monkey D Luffy" (sin punto). Normalizamos ambos y comparamos por
    contención o por palabras, nunca con un substring frágil.
    """
    q, n = _normalizar(consulta), _normalizar(nombre)
    if not q or not n:
        return False
    if q == n or q in n or n in q:
        return True
    palabras = [w for w in q.split() if len(w) >= 3]
    return bool(palabras) and all(w in n.split() for w in palabras)


def _cargar_personajes_op() -> list:
    """Descarga (una sola vez) la lista completa de personajes de One Piece."""
    global _OP_CACHE
    if _OP_CACHE:
        return _OP_CACHE
    url = "https://api.api-onepiece.com/v2/characters/en"
    peticion = urllib.request.Request(
        url, headers={"User-Agent": "keepcoding-adk-lab/1.0"}
    )
    with urllib.request.urlopen(peticion, timeout=25) as respuesta:
        _OP_CACHE = json.load(respuesta)
    return _OP_CACHE


def buscar_personaje_onepiece(nombre: str) -> dict:
    """Busca información de un personaje de One Piece por su nombre.

    Intenta consultar la API publica en vivo de api-onepiece.com. Si la API
    no responde, usa un diccionario de respaldo con personajes principales.

    Args:
        nombre: Nombre del personaje a buscar (por ejemplo "Luffy" o "Zoro").

    Returns:
        Un diccionario con la clave "status" ("ok" o "error"). Si es "ok",
        incluye "personaje" con sus datos y "fuente" ("api" o "fallback").
    """
    # 1) Intento con la API en vivo (lista completa cacheada, filtrada en Python)
    try:
        for p in _cargar_personajes_op():
            if _coincide(nombre, p.get("name") or ""):
                return {
                    "status": "ok",
                    "fuente": "api",
                    "personaje": {
                        "nombre": p.get("name"),
                        "trabajo": p.get("job"),
                        "estado": p.get("status"),
                        "tamaño": p.get("size"),
                        "recompensa": p.get("bounty"),
                    },
                }
    except Exception as error:
        print(f"[tool] API en vivo no disponible ({error}). Uso datos de respaldo.")

    # 2) Respaldo con el diccionario hardcodeado
    for clave, datos in PERSONAJES_FALLBACK.items():
        if _coincide(nombre, clave) or _coincide(nombre, datos["nombre"]):
            return {"status": "ok", "fuente": "fallback", "personaje": datos}

    return {
        "status": "error",
        "mensaje": f"No encontré datos para '{nombre}'.",
        "disponibles": sorted(PERSONAJES_FALLBACK.keys()),
    }


# Prueba rápida: llamamos a la tool como una función Python normal
buscar_personaje_onepiece("Luffy")

{'status': 'ok',
 'fuente': 'api',
 'personaje': {'nombre': 'Monkey D Luffy',
  'trabajo': 'Captain',
  'estado': 'vivant',
  'tamaño': '174cm',
  'recompensa': '3.000.000.000'}}

## 3. El Agent: construir el agente (ejemplo con Gemini)

El **Agent** es el cerebro. Combina un modelo de lenguaje (LLM), unas instrucciones y un conjunto de tools.

Argumentos clave de `Agent`:

- **`name`**: identificador del agente.
- **`model`**: el LLM que razona. Aquí usamos el string `"gemini-2.5-flash"`, un modelo de Google. Crear el agente no llama al modelo, así que esta celda funciona aunque no tengas `GOOGLE_API_KEY`: solo lo ejecutaremos (opcionalmente) en el paso 6.2.
- **`description`**: resumen corto de para qué sirve (útil en sistemas multi-agente).
- **`instruction`**: el prompt de sistema. Define el comportamiento.
- **`tools`**: la lista de funciones que el agente puede llamar. Le pasamos nuestra función directamente.


In [6]:
# Instrucción compartida: la reutilizaremos para los dos modelos
INSTRUCCION = (
    "Eres un experto en el manga y anime One Piece. "
    "Cuando te pregunten por un personaje, usa SIEMPRE la herramienta "
    "buscar_personaje_onepiece para obtener datos fiables. "
    "Responde en español, de forma breve y con tono entusiasta. No uses rayas largas (em dash)."
)

# Agente con un modelo de Google (Gemini)
agente_gemini = Agent(
    name="experto_onepiece_gemini",
    model="gemini-2.5-flash",
    description="Responde preguntas sobre personajes de One Piece usando una tool.",
    instruction=INSTRUCCION,
    tools=[buscar_personaje_onepiece],
)

print(f"Agente creado: {agente_gemini.name}")
print(f"Modelo: {agente_gemini.model}")
print(f"Tools registradas: {[t.__name__ for t in agente_gemini.tools]}")


Agente creado: experto_onepiece_gemini
Modelo: gemini-2.5-flash
Tools registradas: ['buscar_personaje_onepiece']


## 4. ADK es agnóstico de modelo

Aquí está la idea clave del laboratorio.

ADK **no está atado a Gemini**. Para usar un modelo de otro proveedor (OpenAI, Anthropic, etc.) solo cambiamos el argumento `model`: en lugar de un string, le pasamos un objeto **`LiteLlm`**.

`LiteLlm` es el puente que ADK usa para hablar, vía la librería `litellm`, con cientos de modelos distintos. El formato del string es `proveedor/modelo`, por ejemplo `"openai/gpt-5-mini"`.

Observa lo que **NO cambia**:

- la misma `instruction`,
- la misma tool `buscar_personaje_onepiece`,
- más adelante, el mismo `Runner` y la misma `Session`.

Lo único distinto es el `model`. Ese es el corazón de "agnóstico de modelo".


<svg viewBox="0 0 760 320" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="ADK es agnóstico de modelo" style="max-width:760px;width:100%;height:auto;background:#fafafa;border-radius:10px;">
  <style>
    .hdr { font: 700 16px -apple-system, system-ui, sans-serif; fill:#111827; }
    .ttl { font: 600 14px -apple-system, system-ui, sans-serif; fill:#111827; }
    .lbl { font: 400 13px -apple-system, system-ui, sans-serif; fill:#1f2937; }
    .mono { font: 500 12px ui-monospace, "SF Mono", Menlo, monospace; fill:#0f172a; }
    .same { font: 700 11px -apple-system, system-ui, sans-serif; fill:#15803d; letter-spacing:0.5px; }
    .diff { font: 700 11px -apple-system, system-ui, sans-serif; fill:#b91c1c; letter-spacing:0.5px; }
    .box { stroke:#374151; stroke-width:1.5; }
  </style>

  <text x="380" y="28" text-anchor="middle" class="hdr">El mismo agente, dos modelos: solo cambia una línea</text>

  <!-- Left: Gemini -->
  <rect x="30" y="60" width="320" height="230" rx="10" fill="#eff6ff" class="box"/>
  <text x="190" y="85" text-anchor="middle" class="ttl">Agent con Gemini</text>
  <text x="50" y="115" class="mono">name = "experto_onepiece_gemini"</text>
  <text x="50" y="145" class="mono">instruction = INSTRUCCION</text>
  <text x="50" y="175" class="mono">tools = [buscar_personaje_onepiece]</text>
  <rect x="46" y="200" width="290" height="32" rx="6" fill="#fef3c7" stroke="#b45309" stroke-width="1.5"/>
  <text x="56" y="220" class="mono">model = "gemini-2.5-flash"</text>

  <!-- Right: OpenAI -->
  <rect x="410" y="60" width="320" height="230" rx="10" fill="#f0fdf4" class="box"/>
  <text x="570" y="85" text-anchor="middle" class="ttl">Agent con OpenAI</text>
  <text x="430" y="115" class="mono">name = "experto_onepiece_openai"</text>
  <text x="430" y="145" class="mono">instruction = INSTRUCCION</text>
  <text x="430" y="175" class="mono">tools = [buscar_personaje_onepiece]</text>
  <rect x="426" y="200" width="290" height="32" rx="6" fill="#fef3c7" stroke="#b45309" stroke-width="1.5"/>
  <text x="436" y="220" class="mono">model = LiteLlm("openai/gpt-5-mini")</text>

  <!-- Labels SAME / DIFF -->
  <text x="380" y="118" text-anchor="middle" class="same">= IGUAL</text>
  <text x="380" y="148" text-anchor="middle" class="same">= IGUAL</text>
  <text x="380" y="178" text-anchor="middle" class="same">= IGUAL</text>
  <text x="380" y="220" text-anchor="middle" class="diff">ÚNICA DIFERENCIA</text>

  <text x="380" y="275" text-anchor="middle" class="lbl">Misma instruction. Misma tool. Mismo Runner. Misma Session.</text>
  <text x="380" y="293" text-anchor="middle" class="lbl">Solo el argumento <tspan font-family="ui-monospace" font-weight="600">model</tspan> cambia.</text>
</svg>

Mira el dibujo y fíjate en lo que NO cambia (verde) frente a lo único que cambia (rojo). Esta es la promesa central de ADK: tu código de agente queda intacto y puedes intercambiar el LLM debajo.


In [7]:
# Mismo Agent, misma tool, misma instrucción: SOLO cambia el modelo.
# LiteLlm envuelve un modelo de OpenAI usando el formato "proveedor/modelo".
agente_openai = Agent(
    name="experto_onepiece_openai",
    model=LiteLlm(model="openai/gpt-5-mini"),
    description="Responde preguntas sobre personajes de One Piece usando una tool.",
    instruction=INSTRUCCION,
    tools=[buscar_personaje_onepiece],
)

print(f"Agente creado: {agente_openai.name}")
print(f"Tipo de model: {type(agente_openai.model).__name__}")
print("Misma tool y misma instrucción que el agente de Gemini.")


Agente creado: experto_onepiece_openai
Tipo de model: LiteLlm
Misma tool y misma instrucción que el agente de Gemini.


## 5. Runner y Session

Un `Agent` por sí solo no se ejecuta. Necesita dos piezas más:

- **`Session`**: guarda el historial de una conversación (los mensajes y el estado). Una sesión = una conversación. Aquí usamos `InMemorySessionService`, que las guarda en memoria (perfecto para un lab; nada de base de datos).
- **`Runner`**: el motor de ejecución. Recibe un mensaje del usuario, llama al modelo, ejecuta las tools si hace falta y produce **eventos**.

Creamos un helper que, dado un agente, monta su `Runner` y crea una sesión nueva.


In [8]:
APP_NAME = "hola-adk"
USER_ID = "alumno-keepcoding"


async def crear_runner(agente: Agent, session_id: str) -> Runner:
    """Monta un Runner para el agente y crea una sesión nueva."""
    session_service = InMemorySessionService()
    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=session_id,
    )
    return Runner(
        agent=agente,
        app_name=APP_NAME,
        session_service=session_service,
    )


print("Helper crear_runner listo.")


Helper crear_runner listo.


## 6. Ejecutar el agente y leer el stream de eventos

Lanzamos el agente con **`runner.run_async(...)`**. Esto no devuelve una respuesta directa: devuelve un **stream asíncrono de `Event`s**.

Cada `Event` es un paso de lo que pasa por dentro. El stream típico de una pregunta que usa una tool es:

1. El modelo decide llamar a la tool: evento con una **function call**.
2. ADK ejecuta la tool: evento con la **function response** (lo que devolvió nuestra función).
3. El modelo redacta la respuesta final: evento marcado como **final** (`event.is_final_response()`).

Los `Event`s son el **canal de comunicación de ADK**: por ahí viaja todo. El helper siguiente itera el stream e imprime cada evento para que se vea el proceso completo.


### Antes de ejecutarlo: cómo se ve por dentro

Cuando llamas a `runner.run_async(...)` para una pregunta que requiere una tool, el stream de eventos sigue este patrón:

<svg viewBox="0 0 760 540" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Ciclo de vida de los eventos en ADK" style="max-width:760px;width:100%;height:auto;background:#fafafa;border-radius:10px;">
  <defs>
    <marker id="a2" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto">
      <path d="M0,0 L10,5 L0,10 z" fill="#374151"/>
    </marker>
  </defs>
  <style>
    .hdr { font: 700 16px -apple-system, system-ui, sans-serif; fill:#111827; }
    .ttl { font: 600 13px -apple-system, system-ui, sans-serif; fill:#111827; }
    .lbl { font: 400 12px -apple-system, system-ui, sans-serif; fill:#1f2937; }
    .mono { font: 500 12px ui-monospace, "SF Mono", Menlo, monospace; fill:#0f172a; }
    .smono { font: 500 11px ui-monospace, "SF Mono", Menlo, monospace; fill:#374151; }
    .step { font: 700 11px -apple-system, system-ui, sans-serif; fill:#6b7280; letter-spacing:1px; }
    .box { stroke:#374151; stroke-width:1.5; }
    .arr { stroke:#374151; stroke-width:1.5; fill:none; marker-end:url(#a2); }
    .note { font: 400 11px -apple-system, system-ui, sans-serif; fill:#6b7280; font-style: italic; }
  </style>

  <text x="380" y="28" text-anchor="middle" class="hdr">Qué pasa cuando llamas a runner.run_async(...)</text>

  <!-- Step 0: user message -->
  <rect x="200" y="55" width="360" height="50" rx="10" fill="#fef3c7" class="box"/>
  <text x="220" y="73" class="step">TU MENSAJE</text>
  <text x="220" y="93" class="mono">"Cuéntame quién es Zoro y cuál es su sueño."</text>

  <path d="M380,108 L380,135" class="arr"/>

  <!-- Event 1: function_call -->
  <rect x="120" y="140" width="520" height="80" rx="10" fill="#dbeafe" class="box"/>
  <text x="140" y="160" class="step">EVENT 1   ·   author = agent</text>
  <text x="140" y="180" class="ttl">part.function_call</text>
  <text x="140" y="200" class="smono">name: buscar_personaje_onepiece</text>
  <text x="140" y="216" class="smono">args: {"nombre": "Zoro"}</text>
  <text x="620" y="216" text-anchor="end" class="note">el LLM decide llamar a la tool</text>

  <path d="M380,223 L380,250" class="arr"/>

  <!-- ADK runs the tool (local Python) -->
  <rect x="200" y="255" width="360" height="50" rx="10" fill="#f3f4f6" class="box" stroke-dasharray="4,3"/>
  <text x="380" y="276" text-anchor="middle" class="ttl">ADK ejecuta tu función Python en local</text>
  <text x="380" y="294" text-anchor="middle" class="note">(no es un Event, es ejecución real de código)</text>

  <path d="M380,308 L380,335" class="arr"/>

  <!-- Event 2: function_response -->
  <rect x="120" y="340" width="520" height="90" rx="10" fill="#dbeafe" class="box"/>
  <text x="140" y="360" class="step">EVENT 2   ·   author = agent</text>
  <text x="140" y="380" class="ttl">part.function_response</text>
  <text x="140" y="400" class="smono">name: buscar_personaje_onepiece</text>
  <text x="140" y="416" class="smono">response: {"status": "ok", "personaje": {...}}</text>
  <text x="620" y="416" text-anchor="end" class="note">lo que devolvió tu función</text>

  <path d="M380,433 L380,460" class="arr"/>

  <!-- Event 3: final text -->
  <rect x="120" y="465" width="520" height="60" rx="10" fill="#dcfce7" stroke="#15803d" stroke-width="2"/>
  <text x="140" y="485" class="step" fill="#15803d">EVENT 3 (FINAL)   ·   is_final_response() = True</text>
  <text x="140" y="505" class="ttl">part.text</text>
  <text x="140" y="521" class="smono">"Zoro es el espadachín de la tripulación de Luffy..."</text>
</svg>

Tres eventos importantes:

1. **Event 1 · `function_call`**: el LLM ha decidido que necesita la tool y emite la llamada con sus argumentos.
2. **Event 2 · `function_response`**: ADK ha ejecutado tu función Python y emite el resultado.
3. **Event 3 · texto final**: el LLM ya tiene los datos y redacta la respuesta para ti. Aquí `event.is_final_response()` es `True`.

El helper `preguntar()` que viene a continuación itera ese stream e imprime cada evento, así que vas a ver los tres en consola al ejecutar la siguiente celda.


In [9]:
async def preguntar(runner: Runner, session_id: str, pregunta: str) -> None:
    """Envía una pregunta al agente e imprime cada Event del stream."""
    print(f">>> Pregunta: {pregunta}\n")

    mensaje = types.Content(role="user", parts=[types.Part(text=pregunta)])

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session_id,
        new_message=mensaje,
    ):
        # Cada Event lleva un 'author' (quién lo emite) y partes de contenido
        print(f"--- Event de: {event.author} ---")

        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.function_call:
                    fc = part.function_call
                    print(f"  [tool call] {fc.name}(args={dict(fc.args)})")
                elif part.function_response:
                    fr = part.function_response
                    print(f"  [tool response] {fr.name} -> {fr.response}")
                elif part.text:
                    print(f"  [texto] {part.text.strip()}")

        if event.is_final_response():
            print("  (este es el evento FINAL del turno)")
        print()


### 6.1 Ejecutar con el modelo de OpenAI

Probamos primero el agente que usa OpenAI (`gpt-5-mini` vía `LiteLlm`). Necesita la `OPENAI_API_KEY` que configuraste en el setup.


In [10]:
# Montamos el Runner del agente de OpenAI y le hacemos una pregunta
runner_openai = await crear_runner(agente_openai, session_id="sesion-openai")

await preguntar(
    runner_openai,
    session_id="sesion-openai",
    pregunta="Cuéntame quién es Zoro y cuál es su sueño.",
)


>>> Pregunta: Cuéntame quién es Zoro y cuál es su sueño.



--- Event de: experto_onepiece_openai ---
  [tool call] buscar_personaje_onepiece(args={'nombre': 'Zoro'})

--- Event de: experto_onepiece_openai ---
  [tool response] buscar_personaje_onepiece -> {'status': 'ok', 'fuente': 'api', 'personaje': {'nombre': 'Roronoa Zoro', 'trabajo': 'Right-hand man', 'estado': 'living', 'tamaño': '181cm', 'recompensa': '1.101.000.000'}}



--- Event de: experto_onepiece_openai ---
  [texto] ¡Roronoa Zoro! Es el espadachín y mano derecha de Luffy en los Piratas del Sombrero de Paja. Altura 181 cm, está vivo y su recompensa actual es 1.101.000.000 berries.  

Su sueño: convertirse en el mejor espadachín del mundo, cumplir la promesa que hizo a su amiga de la infancia Kuina y, para eso, derrotar al actual mejor espadachín, Dracule Mihawk. Emplea principalmente el estilo de tres espadas, Santoryu.
  (este es el evento FINAL del turno)



> **Qué deberías ver al ejecutar la celda anterior**
>
> El stream imprimirá algo parecido a esto (los textos exactos del LLM cambian entre ejecuciones):
>
> ```text
> >>> Pregunta: Cuéntame quién es Zoro y cuál es su sueño.
>
> --- Event de: experto_onepiece_openai ---
>   [tool call] buscar_personaje_onepiece(args={'nombre': 'Zoro'})
>
> --- Event de: experto_onepiece_openai ---
>   [tool response] buscar_personaje_onepiece -> {'status': 'ok', 'fuente': 'api', 'personaje': {...}}
>
> --- Event de: experto_onepiece_openai ---
>   [texto] Zoro es Roronoa Zoro, espadachín de los Piratas de Sombrero de Paja...
>   (este es el evento FINAL del turno)
> ```
>
> Reconoces los tres eventos del diagrama de arriba: `function_call`, `function_response` y texto final.


### 6.2 La misma pregunta con Gemini (alternativa opcional)

Ahora el agente de Gemini. **El código de ejecución es idéntico**: mismo `preguntar`, mismo tipo de `Runner`, misma tool. Solo cambia el agente que le pasamos, y ese agente solo se diferencia en el `model`.

Esta celda se ejecuta solo si configuraste la `GOOGLE_API_KEY`. Si no, se salta sin error con un aviso: el lab ya está completo con OpenAI.


In [11]:
if os.environ.get("GOOGLE_API_KEY"):
    runner_gemini = await crear_runner(agente_gemini, session_id="sesion-gemini")
    await preguntar(
        runner_gemini,
        session_id="sesion-gemini",
        pregunta="Cuéntame quién es Zoro y cuál es su sueño.",
    )
else:
    print("GOOGLE_API_KEY no configurada: se omite la alternativa con Gemini (es opcional).")
    print("El lab ya ha funcionado con OpenAI. Si quieres probar Gemini, añade la clave en el setup y vuelve a ejecutar esta celda.")


GOOGLE_API_KEY no configurada: se omite la alternativa con Gemini (es opcional).
El lab ya ha funcionado con OpenAI. Si quieres probar Gemini, añade la clave en el setup y vuelve a ejecutar esta celda.


## 7. Recap

En este laboratorio has visto los **cuatro primitivos** de ADK trabajando juntos:

- **Tool**: una función Python tipada con docstring. ADK genera el schema solo, sin escribir JSON.
- **Agent**: el cerebro. Une un modelo (LLM), una `instruction` y una lista de tools.
- **Runner**: el motor que ejecuta el agente y produce un stream de `Event`s.
- **Session**: el historial de una conversación, aquí guardado en memoria con `InMemorySessionService`.

Y la idea central: **ADK es agnóstico de modelo**. El mismo `Agent`, la misma tool, el mismo `Runner` y la misma `Session` funcionan con Gemini o con OpenAI. Lo único que cambia es el argumento `model`: un string para Gemini, o un objeto `LiteLlm("proveedor/modelo")` para el resto.

También viste que los `Event`s son el canal de comunicación de ADK: por ahí pasan las llamadas a tools, sus respuestas y el mensaje final.

Todo esto ha corrido **en local**, en el runtime del notebook. No hemos tocado `adk deploy` ni Vertex AI.

## Retos

Practica modificando este notebook:

1. **Pregunta encadenada**: con el mismo `runner_openai` y la misma `session_id`, lanza una segunda pregunta (por ejemplo "¿Y quién es su capitán?"). Comprueba que la sesión recuerda el contexto.
2. **Personaje inexistente**: pregunta por alguien que no existe (por ejemplo "Háblame de Goku"). Observa cómo la tool devuelve `status: "error"` y cómo el agente reacciona.
3. **Nueva tool**: añade una segunda función, por ejemplo `comparar_recompensas(nombre_a, nombre_b)`, regístrala en `tools=[...]` y pide al agente que compare a dos piratas.
4. **Otro modelo**: cambia `LiteLlm(model="openai/gpt-5-mini")` por otro modelo de OpenAI y observa que nada más del código cambia.
5. **Inspecciona el Event**: dentro de `preguntar`, imprime el objeto `event` completo para ver todos los campos que trae.

## 8. ADK 2.0 · qué cambia (mayo 2026)

Google liberó **ADK 2.0** en GA el 19 de mayo 2026. El cambio mayor es interno: se introduce el **Workflow Runtime**, un motor de ejecución basado en grafo en lugar del hub-and-spoke del 1.x. Para la mayoría de notebooks como este, **el código sigue funcionando sin cambios**: los imports `from google.adk.agents import Agent`, `Runner` y `InMemorySessionService` siguen siendo el API público.

### Lo que sí cambia para el desarrollador

| 1.x | 2.0 |
|---|---|
| `BaseAgent` standalone | `BaseAgent` extiende `BaseNode` (subclase del grafo) |
| Subclases de `Agent` sobrescriben `_run_async_impl` | Inyectas lógica con `BeforeAgentCallback` / `AfterAgentCallback` |
| `Event` schema clásico | `Event` ahora trae campos `node_info` y `output` para el grafo |
| Hub-and-spoke (Runner orquesta lineal) | Workflow Graph (nodos como en LangGraph, pero opinionado) |

### En la práctica

Para este notebook el cambio es **transparente**: instalas `google-adk>=2.0`, el `Agent + Runner + Session` que escribimos sigue funcionando. La diferencia se nota cuando construyes **multi-agente o workflows con ramas**: ahí bajas al Workflow Runtime y compones nodos como en un grafo.

### Cuándo bajas al Workflow Runtime

- Multi-agente coordinado con pasos de validación entre agentes.
- Branches condicionales (si esto, agente A; si lo otro, agente B).
- Human-in-the-loop con nodos de aprobación.

Para el "agente solo con tools" que tienes aquí, el patrón clásico Agent + Runner + Session es la API recomendada en 2.0 también. No hay que reescribir.

### Referencias

- [ADK 2.0 announcement](https://adk.dev/2.0/) · acceso 2026-06-01
- [adk-python v2 workflow samples](https://github.com/google/adk-python/tree/v2/contributing/workflow_samples)

### Conecta con

- **M1 LangGraph** (lab 2) · el Workflow Runtime de ADK 2.0 es conceptualmente lo mismo que LangGraph: agentes como nodos en un grafo con estado.
- **M3 A2A** · ADK 2.0 sigue trayendo A2A nativo, ahora con `Agent Card` exportable directamente desde un nodo del workflow.
- **M5 Harness** · ADK 2.0 incorpora callbacks `Before/After` que son el equivalente "opinionado" del harness que tú construyes a mano.
